# Import libraries


In [0]:
from pyspark.sql.functions import *

# create flag parameter

In [0]:
dbutils.widgets.text('flag_parameter','0')

In [0]:
flag_parameter=dbutils.widgets.get('flag_parameter')

# access the data

In [0]:
df_src=spark.sql('''
                 select distinct(Branch_ID) as Branch_ID,
                  BranchName,1 as dim_branch_key
                  from parquet.`abfss://silver@storagecarsud1.dfs.core.windows.net/cars_data_silver`

                 ''')

# create sink_df

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_branch'): # incremental_Load
     df_sink=spark.sql('''
                      select dim_branch_key,Branch_ID,BranchName 
                      from cars_catalog.gold.dim_branch

                      ''')
else: # Initial Load
    df_sink=spark.sql('''
                      select 1 as dim_branch_key,Branch_ID,BranchName 
                      from parquet.`abfss://silver@storagecarsud1.dfs.core.windows.net/cars_data_silver`
                      where 1=0
                      ''')


In [0]:
df_filter=df_src.join(df_sink,df_src['Branch_ID']==df_sink['Branch_ID'],'left').select(df_src['Branch_ID'],df_src['BranchName']
                                                                                     ,df_sink['dim_branch_key']
                                                                                )

# Filter_Old_Records

In [0]:
df_filter_old=df_filter.filter(df_filter['dim_branch_key'].isNotNull())

# Filter_New_Records

In [0]:
df_filter_new=df_filter.filter(df_filter['dim_branch_key'].isNull())

# create max_value to update surrogate key

In [0]:
if(flag_parameter =='0'):
    max_val=1
else:
    max_val_df=spark.sql("select max(dim_branch_key) from cars_catalog.gold.dim_branch")  
    max_val=max_val_df.collect()[0][0] +1

# Update Surrogate Key

In [0]:
df_filter_new=df_filter_new.withColumn('dim_branch_key',max_val+monotonically_increasing_id())

**Union the old and new data**

In [0]:
df_final=df_filter_new.union(df_filter_old)

# SCD Type-1 (Upsert)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_branch'): # Incremental Load
    delta_table=DeltaTable.forPath(spark,'abfss://gold@storagecarsud1.dfs.core.windows.net/dim_branch')
    
    delta_table.alias("t").merge(df_final.alias('src'),"t.dim_branch_key=src.dim_branch_key")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()

else: # Initial Load
    df_final.write.format('delta')\
        .mode('overwrite')\
        .option('path','abfss://gold@storagecarsud1.dfs.core.windows.net/dim_branch')\
            .saveAsTable('cars_catalog.gold.dim_branch')